In [1]:
### Set parameters
model_name = 'tohoku-bbwwm_vocab-added'

epochs_int = 3
batch_size_int = 128

print(f'{model_name = }')
print(f'{epochs_int = }')
print(f'{batch_size_int = }')

model_name = 'tohoku-bbwwm_vocab-added'
epochs_int = 3
batch_size_int = 128


In [2]:
import os
    
### Set the directory path for output and create the directory
# model_save_path = './tuned_models/' + model_type + '_' + datetime.now().strftime("%Y%m%d_%H%M%S")
model_save_path = './tuned_models/' + model_name + '_jppost-cp-e1to' + str(epochs_int) + '-b' + str(batch_size_int)
checkpoint_save_path = model_save_path + '/checkpoints'

print(f'{model_save_path = }')
print(f'{checkpoint_save_path = }')

os.mkdir(model_save_path)
os.mkdir(checkpoint_save_path)

model_save_path = './tuned_models/tohoku-bbwwm_vocab-added_jppost-cp-e1to3-b128'
checkpoint_save_path = './tuned_models/tohoku-bbwwm_vocab-added_jppost-cp-e1to3-b128/checkpoints'


In [3]:
import json
import pandas as pd
from sentence_transformers import InputExample


def json_dataset_to_df(filepath):
    data_dicts_list = list()
    
    with open(filepath) as fp:
        for line_str in fp:
            line_dict = json.loads(line_str)
            data_dicts_list.append(line_dict)
    
    data_df = pd.DataFrame(data_dicts_list)
    
    return data_df


def create_data_training_list(data_train_df):
    data_train_inputexamples_list = list()
    
    for index, row in data_train_df.iterrows():
        sentence1_str = row['chimei1']
        sentence2_str = row['chimei2']
        label_float = float(row['similarity']) / 5
        data_train_inputexamples_list.append(InputExample(texts=[sentence1_str, sentence2_str], label=label_float))
    
    return data_train_inputexamples_list


def create_data_validation_dict(data_df):
    sentences_1_list = data_df['sentence1'].tolist()
    sentences_2_list = data_df['sentence2'].tolist()
    labels_sr = data_df['label'].astype('float') / 5
    labels_list = labels_sr.tolist()
        
    data_valid_lists_dict = {
        'sentence1': sentences_1_list,
        'sentence2': sentences_2_list,
        'label': labels_list
    }
    return data_valid_lists_dict


In [4]:
# Create datasets for training, validation and test.
import math
from sklearn.model_selection import train_test_split

FILEPATH_INPUT_TRAIN = '../../sbert/jppost/sts_chimei_pairs/sts_chimei_pairs.csv'
FILEPATH_INPUT_VALID = '../../sbert/datasets/jsts-v1.1/valid-v1.1.json'


### Read the training dataset file.
data_train_df = pd.read_csv(FILEPATH_INPUT_TRAIN)
print(data_train_df.head(3))

# Format the data for training.
data_train_inputexamples_list = create_data_training_list(data_train_df)
length_data_train_int = len(data_train_inputexamples_list)
print(length_data_train_int)
print(data_train_inputexamples_list[:3])


### Read the validation dataset file.
data_valid_df = json_dataset_to_df(FILEPATH_INPUT_VALID)
print(data_valid_df.head(3))

# Format the data for training and validation.
data_valid_lists_dict = create_data_validation_dict(data_valid_df)

print(len(data_valid_lists_dict['sentence1']))
print(data_valid_lists_dict['sentence1'][:3])
print(len(data_valid_lists_dict['sentence2']))
print(data_valid_lists_dict['sentence2'][:3])
print(len(data_valid_lists_dict['label']))
print(data_valid_lists_dict['label'][:3])


steps_per_epoch_float = length_data_train_int / batch_size_int

### Calculate warmup steps
warmup_steps_int = steps_per_epoch_float // 100
print(f'{warmup_steps_int = }')

### Calculate checkpoint steps
checkpoint_steps_int = math.ceil(steps_per_epoch_float)
print(f'{checkpoint_steps_int = }')

   Unnamed: 0 chimei1 chimei2  similarity
0           0     北海道     北海道         5.0
1           1     山梨県     山梨県         5.0
2           2     京都府     京都府         5.0
2478651
[<sentence_transformers.readers.InputExample.InputExample object at 0x7f78053edbb0>, <sentence_transformers.readers.InputExample.InputExample object at 0x7f78fe7db280>, <sentence_transformers.readers.InputExample.InputExample object at 0x7f7805371040>]
  sentence_pair_id               yjcaptions_id                    sentence1  \
0                0  100312_421853-104611-31624  レンガの建物の前を、乳母車を押した女性が歩いています。   
1                1        100371-104675-104678             山の上に顔の白い牛が2頭います。   
2                2        100668-104946-104949         バナナを持った人が道路を通行しています。   

                sentence2  label  
0      厩舎で馬と女性とが寄り添っています。    0.0  
1   曇り空の山肌で、牛が２匹草を食んでいます。    2.4  
2  道の上をバナナを背負った男性が歩いています。    3.6  
1457
['レンガの建物の前を、乳母車を押した女性が歩いています。', '山の上に顔の白い牛が2頭います。', 'バナナを持った人が道路を通行しています。']
1457
['厩舎で馬と女性とが寄り添っています。', '曇り空の

In [ ]:
# Get a pre-trained model
from sentence_transformers import SentenceTransformer, losses, models

transformer = models.Transformer(model_name)

pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode='mean')
model = SentenceTransformer(modules=[transformer, pooling], device='cuda')

# Define your train dataset, the dataloader and the train loss
train_loss = losses.CosineSimilarityLoss(model)

In [ ]:
from sentence_transformers import evaluation
from torch.utils.data import DataLoader


dataloader_train = DataLoader(
    dataset=data_train_inputexamples_list,
    batch_size=batch_size_int,
    shuffle=True,  # False
    sampler=None,
    batch_sampler=None,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
    timeout=0,
    worker_init_fn=None,
    prefetch_factor=2,
    persistent_workers=False
)
evaluator = evaluation.EmbeddingSimilarityEvaluator(
    sentences1=data_valid_lists_dict['sentence1'],
    sentences2=data_valid_lists_dict['sentence2'],
    scores=data_valid_lists_dict['label'],
    batch_size=1,  # 16
    main_similarity=evaluation.SimilarityFunction.COSINE,
    name='',
    show_progress_bar=True,
    write_csv=True
)
model.fit(
    train_objectives=[(dataloader_train, train_loss)],
    evaluator=evaluator,
    epochs=epochs_int,
    steps_per_epoch=None,
    scheduler='WarmupLinear',
    warmup_steps=warmup_steps_int,  # 10000
    optimizer_params={'lr': 2e-05},  # <class 'torch.optim.adamw.AdamW'>
    weight_decay=0.01,
    evaluation_steps=0,
    output_path=model_save_path,
    save_best_model=True,
    max_grad_norm=1,
    use_amp=False,
    callback=None,
    show_progress_bar=True,
    checkpoint_path=checkpoint_save_path,  # None
    checkpoint_save_steps=checkpoint_steps_int,  # 500
    checkpoint_save_total_limit=epochs_int  # 0
)